# TabPFN-3.5 at 700,000 training rows

Runs only after a passing pilot and preserves the complete training context during OOM retries.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_REPO_URL='https://github.com/selimsidan/tfm-credit-benchmark.git'
GITHUB_BRANCH='main'
DRIVE_OUTPUT_DIR='/content/drive/MyDrive/tfm_credit_benchmark'
EXPERIMENT_NAME='lending_club_900k'
FORCE_REBUILD_SPLIT=False
FORCE_RERUN=False
RUN_TESTS=True
IGNORE_PILOT_GATE=False
PREDICT_VALIDATION=False
# ==================================================================


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os, subprocess, sys
drive.mount('/content/drive'); repo=Path('/content/tfm-credit-benchmark')
if (repo/'.git').is_dir(): subprocess.run(['git','-C',str(repo),'pull','--ff-only','origin',GITHUB_BRANCH],check=True)
else: subprocess.run(['git','clone','--branch',GITHUB_BRANCH,'--single-branch',GITHUB_REPO_URL,str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
try: tabpfn_token=userdata.get('TABPFN_TOKEN')
except Exception: tabpfn_token=None
if not tabpfn_token: raise RuntimeError('Missing TABPFN_TOKEN. Accept the TabPFN-3.5 license and grant this notebook access to the Colab secret.')
os.environ['TABPFN_TOKEN']=tabpfn_token; tabpfn_token=None
print('Repository commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())


In [ ]:
import json
from tfmbench.config import BenchmarkConfig
from tfmbench.resources import require_tabpfn_full_hardware
from tfmbench.split import verify_manifest
if RUN_TESTS: subprocess.run([sys.executable,'-m','pytest','-q'],check=True)
print(json.dumps(require_tabpfn_full_hardware(),indent=2))
config=BenchmarkConfig(DRIVE_OUTPUT_DIR,EXPERIMENT_NAME); config.validate()
if FORCE_REBUILD_SPLIT:
    from tfmbench.data import prepare_data
    prepare_data(DRIVE_OUTPUT_DIR,EXPERIMENT_NAME,force_rebuild_split=True)
manifest=verify_manifest(config.shared_dir,config); print('Verified manifest:',manifest['manifest_hash'])
pilot_path=config.experiment_root/'pilot'/'pilot_report.json'
if pilot_path.is_file(): print(json.dumps(json.loads(pilot_path.read_text()).get('projections'),indent=2))


In [ ]:
from tfmbench.run import run_method
table=run_method('tabpfn35',700_000,DRIVE_OUTPUT_DIR,EXPERIMENT_NAME,force_rerun=FORCE_RERUN,ignore_pilot_gate=IGNORE_PILOT_GATE,predict_validation=PREDICT_VALIDATION)


In [ ]:
display(table)
print('TabPFN results:',config.experiment_root/'tabpfn35'/'n700000')
